This notebook was tested under the new environment with Python 3.6 and following packages:
- rxnfp==0.1.0
- ipykernel==5.5.6
- numpy==1.19.5
- pandas==1.1.5
- matplotlib==3.2.2
- torch==1.10.2
- tokenizers==0.12.1
- transformers==4.18.0

In [1]:
import pandas as pd
import numpy as np
from rxnfp.transformer_fingerprints import (
    RXNBERTFingerprintGenerator, get_default_model_and_tokenizer
)

model, tokenizer = get_default_model_and_tokenizer()
model.to("cpu")
rxnfp_generator = RXNBERTFingerprintGenerator(model, tokenizer, force_no_cuda=True)
random_seed = 42


/opt/conda/envs/rxnfp/lib/python3.6/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/rxnfp/lib/python3.6/site-packages/torch/cuda/__init__.py:143: UserWarning: 
NVIDIA GeForce RTX 4090 with CUDA capability sm_89 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_37 sm_50 sm_60 sm_70.
If you want to use the NVIDIA GeForce RTX 4090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(incompatible_device_warn.format(device_name, capability, " ".join(arch_list), device_name))


## C–H arylation dataset

In [2]:
rxn_data = pd.read_csv("../dataset/rxn_data/aryl-scope-ligand.csv")
lig_smi_lst, rct1_smi_lst, rct2_smi_lst, pdt_smi_lst = rxn_data['ligand_smiles'].to_list(),rxn_data['electrophile_smiles'].to_list(),rxn_data['nucleophile_smiles'].to_list(),rxn_data['product_smiles'].to_list()
label = rxn_data['yield'].to_numpy()

In [5]:
rxn_smi_with_lig_lst = [f"{rct1}.{rct2}.{lig}>>{pdt}" for rct1, rct2, lig, pdt in zip(rct1_smi_lst, rct2_smi_lst, lig_smi_lst, pdt_smi_lst)]
rxn_fp_arr_with_lig = np.array(rxnfp_generator.convert_batch(rxn_smi_with_lig_lst))
rxn_smi_wo_lig_lst = [f"{rct1}.{rct2}>>{pdt}" for rct1, rct2, pdt in zip(rct1_smi_lst, rct2_smi_lst, pdt_smi_lst)]
rxn_fp_arr_wo_lig = np.array(rxnfp_generator.convert_batch(rxn_smi_wo_lig_lst))

In [6]:
rxn_smi_with_lig_rxnfp_map = {smi:fp for smi, fp in zip(rxn_smi_with_lig_lst, rxn_fp_arr_with_lig)}
rxn_smi_wo_lig_rxnfp_map = {smi:fp for smi, fp in zip(rxn_smi_wo_lig_lst, rxn_fp_arr_wo_lig)}

In [8]:
np.save("./gen_desc/aryl_scope_rxn_smi_with_lig_rxnfp_map.npy", rxn_smi_with_lig_rxnfp_map)
np.save("./gen_desc/aryl_scope_rxn_smi_wo_lig_rxnfp_map.npy", rxn_smi_wo_lig_rxnfp_map)

## Asymmetric thiol addition dataset

In [9]:
rxn_data = pd.read_csv("../dataset/rxn_data/NS_acetal_dataset_with_pdt.csv")
rxn_data

,Unnamed: 0,Imine,Thiol,Catalyst,ΔΔG,Product
0,0,O=C(/N=C/c1ccccc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.179891,O=C(NC(Sc1ccccc1)c1ccccc1)c1ccccc1
1,1,O=C(/N=C/c1ccccc1)c1ccccc1,CCS,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,0.501759,CCSC(NC(=O)c1ccccc1)c1ccccc1
2,2,O=C(/N=C/c1ccccc1)c1ccccc1,SC1CCCCC1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,0.650584,O=C(NC(SC1CCCCC1)c1ccccc1)c1ccccc1
3,3,O=C(/N=C/c1ccccc1)c1ccccc1,COc1ccc(S)cc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.238109,COc1ccc(SC(NC(=O)c2ccccc2)c2ccccc2)cc1
4,4,O=C(/N=C/c1ccc(C(F)(F)F)cc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.179891,O=C(NC(Sc1ccccc1)c1ccc(C(F)(F)F)cc1)c1ccccc1
...,...,...,...,...,...,...
1070,1070,O=C(/N=C/c1ccccc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.531803,O=C(NC(Sc1ccccc1)c1ccccc1)c1ccccc1
1071,1071,O=C(/N=C/c1ccccc1)c1ccccc1,Cc1ccccc1S,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.531803,Cc1ccccc1SC(NC(=O)c1ccccc1)c1ccccc1
1072,1072,O=C(/N=C/c1ccc(C(F)(F)F)cc1)c1ccccc1,Cc1ccccc1S,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.370104,Cc1ccccc1SC(NC(=O)c1ccccc1)c1ccc(C(F)(F)F)cc1
1073,1073,O=C(/N=C/c1cccc2ccccc12)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.301167,O=C(NC(Sc1ccccc1)c1cccc2ccccc12)c1ccccc1


In [10]:
imine_lst = rxn_data['Imine'].to_list()
thiol_lst = rxn_data['Thiol'].to_list()
cat_lst = rxn_data['Catalyst'].to_list()
pdt_lst = rxn_data['Product'].to_list()
label = rxn_data['ΔΔG'].to_numpy()

In [11]:
rxn_smi_with_cat_lst = [f"{imine}.{thiol}.{cat}>>{pdt}" for imine,thiol,cat,pdt in zip(imine_lst,thiol_lst,cat_lst,pdt_lst)]
rxn_fp_arr_with_cat = np.array(rxnfp_generator.convert_batch(rxn_smi_with_cat_lst))

rxn_smi_wo_cat_lst = [f"{imine}.{thiol}>>{pdt}" for imine,thiol,pdt in zip(imine_lst,thiol_lst,pdt_lst)]
rxn_fp_arr_wo_cat = np.array(rxnfp_generator.convert_batch(rxn_smi_wo_cat_lst))

In [12]:
rxn_smi_with_cat_rxnfp_map = {smi:fp for smi, fp in zip(rxn_smi_with_cat_lst, rxn_fp_arr_with_cat)}
rxn_smi_wo_cat_rxnfp_map = {smi:fp for smi, fp in zip(rxn_smi_wo_cat_lst, rxn_fp_arr_wo_cat)}

In [13]:
np.save("./gen_desc/thiol_add_rxn_smi_with_cat_rxnfp_map.npy", rxn_smi_with_cat_rxnfp_map)
np.save("./gen_desc/thiol_add_rxn_smi_wo_cat_rxnfp_map.npy", rxn_smi_wo_cat_rxnfp_map)